In [2]:
from openai import OpenAI

In [ ]:
client = OpenAI()


In [3]:

documents = [
    {
        "document_id": "AR_POLICY",
        "title": "Accounts Receivable Policy",
        "chunks": [
            "Customers are generally invoiced on Net 30 payment terms unless a contract specifies otherwise.",
            "Payments received from customers must be applied to the correct open invoice. If a payment is received but remains unapplied, the accounts receivable team should investigate the remittance details and update the payment application.",
            "Past-due balances should be reviewed by the collections team. Customers with overdue balances may require follow-up before additional credit is extended.",
            "Customer credit balances may be applied to future invoices or refunded according to company policy."
        ]
    },
    {
        "document_id": "INTEGRATION_RUNBOOK",
        "title": "Invoice Integration Runbook",
        "chunks": [
            "Invoice payloads are transmitted to the external API after source records pass validation.",
            "Requests that exceed the external service payload-size limit may fail. Large embedded images or attachments should be removed or handled separately before the request is retried.",
            "Transient API timeouts should be retried with bounded retry logic and exponential backoff.",
            "Duplicate invoice records should be prevented by using a stable unique source identifier and enforcing idempotent processing."
        ]
    },
    {
        "document_id": "PAYROLL_POLICY",
        "title": "Payroll Allocation Policy",
        "chunks": [
            "Employee payroll expense is allocated to service divisions using hours worked during the corresponding pay period.",
            "Paid time off is allocated using historical service-division work patterns when no current-period labor hours are available.",
            "Bonuses are normalized separately from regular wages before reporting labor cost by division.",
            "Payroll records should be reconciled to the source payroll system before they are published to reporting."
        ]
    }
]

In [4]:
#Flatten the documents

chunks = []

for document in documents:
    for chunk_id, text in enumerate(document["chunks"]):
        chunks.append({
            "document_id": document["document_id"],
            "document_title": document["title"],
            "chunk_id": chunk_id,
            "text": text
        })

In [8]:
for chunk in chunks:
    print(chunk)

{'document_id': 'AR_POLICY', 'document_title': 'Accounts Receivable Policy', 'chunk_id': 0, 'text': 'Customers are generally invoiced on Net 30 payment terms unless a contract specifies otherwise.'}
{'document_id': 'AR_POLICY', 'document_title': 'Accounts Receivable Policy', 'chunk_id': 1, 'text': 'Payments received from customers must be applied to the correct open invoice. If a payment is received but remains unapplied, the accounts receivable team should investigate the remittance details and update the payment application.'}
{'document_id': 'AR_POLICY', 'document_title': 'Accounts Receivable Policy', 'chunk_id': 2, 'text': 'Past-due balances should be reviewed by the collections team. Customers with overdue balances may require follow-up before additional credit is extended.'}
{'document_id': 'AR_POLICY', 'document_title': 'Accounts Receivable Policy', 'chunk_id': 3, 'text': 'Customer credit balances may be applied to future invoices or refunded according to company policy.'}
{'doc

In [6]:
#create embeddings for chunk texts and add the embedding to the chunk object
chunk_texts = [
    chunk["text"]
    for chunk in chunks
]

response = client.embeddings.create(
    model="text-embedding-3-small",
    input=chunk_texts
)

for chunk, item in zip(chunks, response.data):
    chunk["embedding"] = item.embedding

In [7]:
#Build FAISS index
#Create embedding matrix array of embeddings
#normalize the embeddings
#create the index using hnsw

import numpy as np
import faiss

embedding_matrix = np.array(
    [chunk["embedding"] for chunk in chunks],
    dtype="float32"
)

faiss.normalize_L2(embedding_matrix)

dimension = embedding_matrix.shape[1]

index = faiss.IndexHNSWFlat(
    dimension,
    32,
    faiss.METRIC_INNER_PRODUCT
)

index.add(embedding_matrix)

In [8]:
#Chunk search function
def search_chunks(query: str, top_k: int = 3):

    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=query
    )

    query_vector = np.array(
        [response.data[0].embedding],
        dtype="float32"
    )

    faiss.normalize_L2(query_vector)

    scores, indices = index.search(
        query_vector,
        top_k
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):
        chunk = chunks[idx]

        results.append({
            "score": float(score),
            "document_id": chunk["document_id"],
            "document_title": chunk["document_title"],
            "chunk_id": chunk["chunk_id"],
            "text": chunk["text"]
        })

    return results



In [28]:
query = "How should PTO be assigned when there are no current labor hours?"

results = search_chunks(query)

for result in results:
    print(
        f"{result['score']:.4f} | "
        f"{result['document_id']} | "
        f"chunk {result['chunk_id']}"
    )
    print(result["text"])
    print()

0.6400 | PAYROLL_POLICY | chunk 1
Paid time off is allocated using historical service-division work patterns when no current-period labor hours are available.

0.4482 | PAYROLL_POLICY | chunk 0
Employee payroll expense is allocated to service divisions using hours worked during the corresponding pay period.

0.3159 | PAYROLL_POLICY | chunk 2
Bonuses are normalized separately from regular wages before reporting labor cost by division.



In [4]:
import tiktoken

In [6]:
encoding = tiktoken.get_encoding("cl100k_base")

def count_tokens(text: str) -> int:
    return len(encoding.encode(text))


In [7]:
def chunk_by_paragraphs(
    text: str,
    max_tokens: int = 300
):
    paragraphs = [
        p.strip()
        for p in text.split("\n\n")
        if p.strip()
    ]

    chunks = []
    current_chunk = []
    current_tokens = 0

    for paragraph in paragraphs:

        paragraph_tokens = count_tokens(paragraph)

        # If the paragraph itself is too large,
        # handle it separately.
        if paragraph_tokens > max_tokens:

            # Save anything we've already accumulated.
            if current_chunk:
                chunks.append(
                    "\n\n".join(current_chunk)
                )

                current_chunk = []
                current_tokens = 0

            # Fall back to token-based splitting
            # for this oversized paragraph.
            paragraph_chunks = split_large_paragraph(
                paragraph,
                max_tokens=max_tokens
            )

            chunks.extend(paragraph_chunks)
            continue

        # Would adding this paragraph exceed the ceiling?
        if (
            current_chunk
            and current_tokens + paragraph_tokens > max_tokens
        ):
            chunks.append(
                "\n\n".join(current_chunk)
            )

            current_chunk = []
            current_tokens = 0

        current_chunk.append(paragraph)
        current_tokens += paragraph_tokens

    # Don't forget the last chunk.
    if current_chunk:
        chunks.append(
            "\n\n".join(current_chunk)
        )

    return chunks

In [8]:
def chunk_by_paragraphs_with_overlap(
        text: str,
        max_tokens: int = 300,
        overlap_tokens: int = 50
):
    paragraphs = [
        p.strip()
        for p in text.split("\n\n")
        if p.strip()
    ]

    chunks = []
    current_chunk = []
    current_tokens = 0

    for paragraph in paragraphs:
        paragraph_tokens = count_tokens(paragraph)

        if paragraph_tokens > max_tokens:
            if current_chunk:
                chunks.append("\n\n".join(current_chunk))
            paragraph_chunks = split_large_paragraph(
                paragraph,
                max_tokens=max_tokens
            )
            chunks.extend(paragraph_chunks)

            current_chunk = []
            current_tokens = 0
            continue

        if(
            current_chunk
            and current_tokens + paragraph_tokens > max_tokens
        ):
            chunks.append("\n\n".join(current_chunk))
            #Carry trailing paragraphs forward as overlap
            # up to overlap_tokens worth of content
            overlap_paragraphs = []
            overlap_tokens_used = 0

            for p in reversed(current_chunk):
                p_tokens = count_tokens(p)
                if overlap_tokens_used + p_tokens > overlap_tokens:
                    break
                overlap_paragraphs.insert(0, p)
                overlap_tokens_used += p_tokens
            current_chunk = overlap_paragraphs
            current_tokens = overlap_tokens_used

        current_chunk.append(paragraph)
        current_tokens += paragraph_tokens
    if current_chunk:
        chunks.append("\n\n".join(current_chunk))
    return chunks


In [9]:
def split_large_paragraph(
    paragraph: str,
    max_tokens: int = 300,
    overlap: int = 50
):
    tokens = encoding.encode(paragraph)

    chunks = []

    start = 0

    while start < len(tokens):

        end = start + max_tokens

        chunk_tokens = tokens[start:end]

        chunks.append(
            encoding.decode(chunk_tokens)
        )

        start += max_tokens - overlap

    return chunks

In [10]:
document_text = """
Accounts Receivable Policy

Customers are generally invoiced on Net 30 payment terms unless otherwise specified in the governing contract.

Payments received from customers must be applied to the correct open invoice. Remittance information should be reviewed before payment application.

If a payment has been received but remains unapplied, the accounts receivable team should investigate the payment reference, customer account, and invoice details before updating the transaction.

Past-due customer balances should be reviewed by the collections team. Follow-up should occur according to the aging status and customer relationship.

Customers with unresolved overdue balances may require additional approval before further credit is extended.

Customer credit balances may be applied against future invoices when appropriate. Refunds must follow the company's approval process.
"""

In [11]:
no_overlap = chunk_by_paragraphs(document_text, max_tokens=80)

In [19]:
with_overlap = chunk_by_paragraphs_with_overlap(document_text, max_tokens=80, overlap_tokens=50)

In [20]:
for label, result in [("NO OVERLAP", no_overlap), ("WITH OVERLAP", with_overlap)]:
    print(f"\n==== {label} ====")
    for i, chunk in enumerate(result):
        print(f"\n--- Chunk {i} ({count_tokens(chunk)} tokens) ----")
        print(chunk)


==== NO OVERLAP ====

--- Chunk 0 (81 tokens) ----
Accounts Receivable Policy

Customers are generally invoiced on Net 30 payment terms unless otherwise specified in the governing contract.

Payments received from customers must be applied to the correct open invoice. Remittance information should be reviewed before payment application.

If a payment has been received but remains unapplied, the accounts receivable team should investigate the payment reference, customer account, and invoice details before updating the transaction.

--- Chunk 1 (63 tokens) ----
Past-due customer balances should be reviewed by the collections team. Follow-up should occur according to the aging status and customer relationship.

Customers with unresolved overdue balances may require additional approval before further credit is extended.

Customer credit balances may be applied against future invoices when appropriate. Refunds must follow the company's approval process.

==== WITH OVERLAP ====

--- Chunk 0

In [22]:
for i, chunk in enumerate(document_chunks):
    print(f"\n--- Chunk {i} ---")
    print(f"Tokens: {count_tokens(chunk)}")
    print(chunk)


--- Chunk 0 ---
Tokens: 81
Accounts Receivable Policy

Customers are generally invoiced on Net 30 payment terms unless otherwise specified in the governing contract.

Payments received from customers must be applied to the correct open invoice. Remittance information should be reviewed before payment application.

If a payment has been received but remains unapplied, the accounts receivable team should investigate the payment reference, customer account, and invoice details before updating the transaction.

--- Chunk 1 ---
Tokens: 63
Past-due customer balances should be reviewed by the collections team. Follow-up should occur according to the aging status and customer relationship.

Customers with unresolved overdue balances may require additional approval before further credit is extended.

Customer credit balances may be applied against future invoices when appropriate. Refunds must follow the company's approval process.


In [ ]:
chunks = []

for chunk_id, text in enumerate(document_chunks):
    chunks.append({
        "document_id": "AR_POLICY",
        "document_title": "Accounts Receivable Policy",
        "chunk_id": chunk_id,
        "text": text
    })

: 

In [23]:
faithfulness = { 
        'score':1 ,
        'checks' : [
            {'claim': 'A source record must contain a valid customer identifier.', 'label': 'supported', 'evidence': 'confirmation that a valid customer identifier'}, 
            {'claim': 'A source record must contain an invoice number.', 'label': 'unsupported', 'evidence': 'confirmation that a valid customer identifier, invoice number, and line-item total are present on the record.'}, 
            {'claim': 'A source record must contain a line-item total.', 'label': 'supported', 'evidence': 'including confirmation that a valid customer identifier, invoice number, and line-item total are present on the record.'}, 
            {'claim': 'The invoice payload is sent to the external API.', 'label': 'supported', 'evidence': 'Invoice payloads are transmitted to the external API'}
        ]
    }


In [18]:
print(type(faithfulness))
print(faithfulness)

<class 'dict'>
{'score': 1, 'checks': [{'claim': 'A source record must contain a valid customer identifier.', 'label': 'supported', 'evidence': 'confirmation that a valid customer identifier'}, {'claim': 'A source record must contain an invoice number.', 'label': 'supported', 'evidence': 'confirmation that a valid customer identifier, invoice number, and line-item total are present on the record.'}, {'claim': 'A source record must contain a line-item total.', 'label': 'supported', 'evidence': 'including confirmation that a valid customer identifier, invoice number, and line-item total are present on the record.'}, {'claim': 'The invoice payload is sent to the external API.', 'label': 'supported', 'evidence': 'Invoice payloads are transmitted to the external API'}]}


In [19]:
print(type(faithfulness['checks']))

<class 'list'>


In [24]:
for i in faithfulness['checks']:
    if i.get('label') == "unsupported":
        print(f"unsupported evidence found: {i.get('claim')}")

unsupported evidence found: A source record must contain an invoice number.
